# Combined extraction packages

**Lesson 7 -- Module 1: Extraction**

So far we've assembled our own extraction pipeline from separate tools -- PyMuPDF for text layers, Tesseract for OCR, and custom logic to decide which to use. An alternative approach is to use a **combined extraction package** that handles all of this in one tool.

Several packages take this all-in-one approach:
- [**Docling**](https://github.com/docling-project/docling) -- IBM's open-source document understanding library
- [**Unstructured**](https://github.com/Unstructured-IO/unstructured) -- open-source document processing with cloud and local modes
- [**Marker**](https://github.com/VikParuchuri/marker) -- converts PDFs to markdown using deep learning models
- **Cloud APIs** -- Azure Document Intelligence, Google Document AI, AWS Textract

We'll use **Docling** as our example. It's open-source, runs locally, and handles OCR automatically -- using the native Vision framework on macOS, or EasyOCR on Linux and in Codespaces.

## What you'll learn

- How combined extraction packages differ from modular tools
- How to use Docling for text extraction and OCR
- When to force full-page OCR vs trusting the existing text layer
- The speed vs quality tradeoff between combined and modular approaches

## 1. Install Dependencies

In [1]:
%pip install docling -q

Note: you may need to restart the kernel to use updated packages.


## 2. Imports & Configuration

In [1]:
import html
import logging
import platform
import time
from pathlib import Path

from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import (
    DocumentConverter, PdfFormatOption
)

# macOS uses the native Vision framework; Linux/Codespace uses EasyOCR
if platform.system() == "Darwin":
    from docling.datamodel.pipeline_options import OcrMacOptions
    OCR_OPTIONS = OcrMacOptions()
else:
    from docling.datamodel.pipeline_options import EasyOcrOptions
    OCR_OPTIONS = EasyOcrOptions()

logging.getLogger("docling_core").setLevel(logging.ERROR)

PDF_DIR = Path("enron_pdfs")

# Reference PDFs used throughout the notebook
CLEAN_PDF  = PDF_DIR / "E0048ADF3.pdf"   # clean digital text layer
SAMPLE_PDF = PDF_DIR / "E61D04918.pdf"   # scanned, moderate OCR errors
EMPTY_PDF  = PDF_DIR / "E0033CF3B.pdf"   # no text layer at all

pdf_files = sorted(PDF_DIR.glob("*.pdf"))
print(f"Found {len(pdf_files):,} PDFs")
print(f"OCR backend: {type(OCR_OPTIONS).__name__}")

Found 4,911 PDFs
OCR backend: OcrMacOptions


## 3. Default Extraction

Docling's `DocumentConverter` handles the full pipeline: layout analysis, text extraction, and OCR if needed. We disable table structure detection and picture generation since our email PDFs don't need them.

This is typical of combined packages — you configure what you need, and the tool handles the rest.

In [2]:
pipeline_options = PdfPipelineOptions()
pipeline_options.do_ocr = True
pipeline_options.do_table_structure = False
pipeline_options.generate_picture_images = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

In [3]:
# Pre-load all Docling models now so the first convert() call doesn't stall
# This takes 30-60 seconds — you'll see "Models loaded" when it's done
print("Loading Docling models (layout analysis, OCR, etc.)...")
converter.initialize_pipeline(InputFormat.PDF)
print("Models loaded.")

Loading Docling models (layout analysis, OCR, etc.)...
Models loaded.


In [4]:
result = converter.convert(str(SAMPLE_PDF))
default_text = html.unescape(result.document.export_to_text())

print("=== Default Docling (trusts text layer) ===")
print(default_text[:1000])

=== Default Docling (trusts text layer) ===
CONFIDENTIAL

Enron Corp.

Case No. EC-26002-016038

Doc No. Es6lbo4918

Bate: O1/15/2003

ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.

RELEASE IN PART

From:

Ker i Thompson <ker i.thompson@enron,.

com>

Sent:

Wed, 22 Nov 2000 05:41:00 -0800 (PST)

To:

Kate Symes <kate.symes@enron.com>

Subject :

natsource checkout

465810

mat strike price 70.00

8.00 premium

465556

bob broker has midc

SOAR DDERT DAL

Enron Gerp.

Case Ro. 20-7 002-91078

wea No. RETDC4I9le2

pater

Cle 5/7003

ERRGN TORP. - PRODUCED PURSUANT TO SEAC SUBPORNA.

SLBCECT TC PROTROLDIVE ORGAR. CORE OCSANYTIAL TREATKERT ReOLE

ES Tsis.

EKRON-S206 741254888

CONFIDENTIAL Enron Corp. Case No. EC-2002-01038 Doc No. E61D04918 Date: 01/15/2003 ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. RELEASE IN PART

From:

Sent:

To:

Subject:

Kerri 

By default, Docling trusts the existing text layer. Since `E61D04918.pdf` has a noisy OCR layer (`Bate:` for `Date:`, `ENRON CORE.` for `ENRON CORP.`), the output carries those same errors.

This is the same behavior as PyMuPDF's `get_text()`. To get better text, we need to tell Docling to ignore the text layer and OCR the page image directly.

**A note on angle brackets:** Docling's `export_to_text()` internally calls its markdown exporter, which HTML-encodes `<` and `>` as `&lt;` and `&gt;`. Since our email headers contain `Name <email>` format, we wrap the output in `html.unescape()` to restore the original characters.


## 4. Forcing full-page OCR

Setting `force_full_page_ocr=True` tells Docling to ignore the existing text layer and re-OCR from the page image.

On macOS, Docling uses the native **Vision framework** -- no Tesseract install required. On Linux, it falls back to EasyOCR.

> This creates a second `DocumentConverter` with different options. Each instance loads its own copy of the models -- fine for experimentation, but in a real pipeline you'd reuse a single converter to avoid duplicating model memory.

In [5]:
pipeline_options_ocr = PdfPipelineOptions()
pipeline_options_ocr.do_ocr = True
pipeline_options_ocr.do_table_structure = False
pipeline_options_ocr.generate_picture_images = False

# Use the platform-appropriate OCR backend with forced full-page OCR
if platform.system() == "Darwin":
    pipeline_options_ocr.ocr_options = OcrMacOptions(force_full_page_ocr=True)
else:
    pipeline_options_ocr.ocr_options = EasyOcrOptions(force_full_page_ocr=True)

converter_ocr = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options_ocr
        )
    }
)

converter_ocr.initialize_pipeline(InputFormat.PDF)
print("OCR converter ready.")

OCR converter ready.


In [6]:
result = converter_ocr.convert(str(SAMPLE_PDF))
ocr_text = html.unescape(result.document.export_to_text())

print("=== Docling with force_full_page_ocr=True ===")
print(ocr_text[:1000])

=== Docling with force_full_page_ocr=True ===
CONFIDENTIAL Enron Corp. Case No. EC-2002-01038 Doc No. E61D04918 Date: 01/15/2003 ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. RELEASE IN PART

From:

Sent:

To:

Subject:

Kerri Thompson <kerri.thompson@enron.com>

Wed, 22 Nov 2000 05:41:00 -0800 (PST)

Kate Symes <kate.symes@enron.com>

natsource checkout

465810 matt strike price 70.00 8.00 premium

465556 bob broker has mid c

ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.


Compare this output to the default extraction above. Forced OCR re-reads the page image using the macOS Vision framework (or EasyOCR on Linux) -- a different engine from the Tesseract that produced the original text layer.

Whether the output is better depends on the source image quality. On B&W scans, different OCR engines may produce different errors rather than strictly better text. Check the header fields (Case No, Doc No, Date) to see where the engines disagree.

## 5. OCR on Empty PDFs

What about PDFs with no text layer at all? Docling handles these automatically — when it finds no text layer, it OCRs the page image.

In [7]:
result = converter.convert(str(EMPTY_PDF))
empty_text = html.unescape(result.document.export_to_text())

print(f"=== {EMPTY_PDF.name} (no text layer) ===")
print(empty_text[:1000])

=== E0033CF3B.pdf (no text layer) ===
CONFIDENTIAL Enron Corp. Case No. EC-2002-01038 Doc No. E0033CF3B Date: 01/15/2003 ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. RELEASE IN PART From: Sent: To: Cc: Subject: Schedule Crawler <pete.davis@enron.com> Mon, 9 Apr 2001 01:41:00 -0700 (PDT) pete.davis@enron.com bert.meyers@enron.com,bill.williams.III@enron.com, Craig.Dean@enron.com,dporter3@enron.com, Eric.Linder@enron.com Geir. Solberg@enron.com, jbryson@enron.com,leaf.harasin@enron.com monika.causholli@enron.com, mark.guzman@enron.com, pete.davis@enron.com, ryan.slinger@enron.com Start Date: 4/9/01; HourAhead hour: 9; <CODESITE> Start Date: 4/9/01; HourAhead hour: 9; No ancillary schedules awarded. Variances detected. Variances detected in SC Trades schedule. LOG MESSAGES: PARSING FILE -->> O: \Portland\WestDesk\California Scheduling\ISO | Schedules \2001040909.txt ---- Energy Import/Export Schedule ---*** Final schedul

Docling detects there's no text layer and OCRs automatically — no special configuration needed.

## 6. One Tool for Any PDF

With PyMuPDF + Tesseract, we needed a tiered strategy: check the file size, use `get_text()` for clean files, fall back to Tesseract CLI for the rest. Two different code paths, two different tools.

This is the core appeal of combined packages — the tool handles the decision-making automatically. Docling reads the text layer where one exists, and OCRs the page image where it doesn't. One call for any PDF:

In [8]:
result = converter.convert(str(CLEAN_PDF))
clean_text = html.unescape(result.document.export_to_text())

print(f"=== {CLEAN_PDF.name} (clean digital text) ===")
print(clean_text[:1000])

=== E0048ADF3.pdf (clean digital text) ===
CONFIDENTIAL Enron Corp. Case No. EC-2002-01038 Doc No. E0048ADF3 Date: 01/15/2003 ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. RELEASE IN FULL

From: Nemec, Gerald <gerald.nemec@enron.com> Sent: Wed, 17 Oct 2001 13:11:22 -0700 (PDT) To: Lucci, Paul T. <t..lucci@enron.com> Cc: Tycholiz, Barry <barry.tycholiz@enron.com>, Williams, Jason R (Credit) <.williams@enron.com> Subject: Kennedy Amendment

Here is the latest.

CONFIDENTIAL Enron Corp. Case No. EC-2002-01038 Doc No. E0048ADF3 Date: 01/15/2003 ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA. SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED. ENRON-462405886129


The default converter reads the text layer on clean files and OCRs image-only files -- no file-size heuristic, no switching tools.

The one limitation: it **trusts** existing text layers, so files with OCR errors carry those errors through (as we saw in section 3). For those, you need `force_full_page_ocr`. But for a corpus where most files are either clean or image-only, one converter handles everything in a single code path.

## 7. Quality comparison

Let's compare Docling's forced OCR output against PyMuPDF and Tesseract on the same file.

In [9]:
import pymupdf
import subprocess
import tempfile

# PyMuPDF get_text()
doc = pymupdf.open(SAMPLE_PDF)
page = doc[0]
pymupdf_text = page.get_text()

# Tesseract CLI at 300 DPI
pix = page.get_pixmap(dpi=300)
with tempfile.NamedTemporaryFile(suffix=".png") as f:
    pix.save(f.name)
    result = subprocess.run(
        ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
        capture_output=True, text=True,
    )
    tesseract_text = result.stdout
doc.close()

print("=" * 60)
print("PyMuPDF get_text() — existing OCR layer")
print("=" * 60)
print(pymupdf_text[:400])

print(f"\n{'=' * 60}")
print("Tesseract CLI — re-OCR at 300 DPI")
print("=" * 60)
print(tesseract_text[:400])

print(f"\n{'=' * 60}")
print("Docling — force_full_page_ocr (macOS Vision)")
print("=" * 60)
print(ocr_text[:400])

PyMuPDF get_text() — existing OCR layer
CONFIDENTIAL
Enron Corp.
Case No. EC-26002-016038
Doc No. Es6lbo4918
Bate: O1/15/2003
ENRON CORE. - PRODUCED PURSUANT TO FERC SUBPOENA.
SUBJECT TO EROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART
From:
Kerri Thompson <kerri.thompson@enron,.
com>
Sent:
Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To:
Kate Symes <kate.symes@enron.com>
Subject :
natsource checkout
465810
matt
strike price

Tesseract CLI — re-OCR at 300 DPI
CONFIDENTIAL

Enron Corp.

Case No. EC-2002-01038

Doc No. E61D04918

Date: 01/15/2003

ENRON CORP. - PRODUCED PURSUANT TO FERC SUBPOENA.

SUBJECT TO PROTECTIVE ORDER. CONFIDENTIAL TREATMENT REQUESTED.
RELEASE IN PART

From: Kerri Thompson <kerri.thompson@enron.com>
Sent: Wed, 22 Nov 2000 05:41:00 -0800 (PST)
To: Kate Symes <kate.symes@enron.com>
Subject: natsource checkout

465810

matt

strike p

Docling — force_full_page_ocr (macOS Vision)
CONFIDENTIAL Enron Corp. Case No. EC-2002-01038 Doc No. E61D04918 Date: 01/15

### Observations

Each OCR engine has its own error profile. The existing text layer, Tesseract re-OCR, and Docling's Vision OCR will each get different things right and wrong on the same image.

Our scans are generally quite good, but no engine is perfect on degraded scans.

## 8. Speed comparison -- combined vs modular

How does a combined package compare when used as the complete extraction tool vs our modular PyMuPDF + Tesseract tiered strategy?

We'll test three approaches on a mixed sample of clean and scanned PDFs:

In [10]:
import pymupdf
import subprocess
import tempfile

# Build a mixed sample: 10 digital (no images) + 10 scanned (has images)
digital_sample = []
scanned_sample = []
for p in pdf_files:
    doc = pymupdf.open(p)
    has_images = bool(doc[0].get_images())
    doc.close()
    if has_images and len(scanned_sample) < 10:
        scanned_sample.append(p)
    elif not has_images and len(digital_sample) < 10:
        digital_sample.append(p)
    if len(digital_sample) >= 10 and len(scanned_sample) >= 10:
        break

sample = digital_sample + scanned_sample
n = len(sample)

# --- PyMuPDF + Tesseract per-page strategy ---
start = time.perf_counter()
for p in sample:
    doc = pymupdf.open(p)
    for page in doc:
        text = page.get_text().strip()
        if not text:
            pix = page.get_pixmap(dpi=300)
            with tempfile.NamedTemporaryFile(suffix=".png") as f:
                pix.save(f.name)
                subprocess.run(
                    ["tesseract", f.name, "stdout", "--psm", "6", "-l", "eng"],
                    capture_output=True, text=True,
                )
    doc.close()
elapsed_tiered = time.perf_counter() - start

# --- Docling default (automatic) ---
start = time.perf_counter()
for p in sample:
    result = converter.convert(str(p))
    _ = result.document.export_to_text()
elapsed_default = time.perf_counter() - start

# --- Docling force_full_page_ocr ---
start = time.perf_counter()
for p in sample:
    result = converter_ocr.convert(str(p))
    _ = result.document.export_to_text()
elapsed_force = time.perf_counter() - start

print(f"Mixed sample: {n} PDFs (10 digital + 10 scanned)")
print()
print(f"{'Method':<40} {'Time':>8} {'PDFs/sec':>10}")
print("-" * 60)
print(f"{'PyMuPDF + Tesseract (per-page)':<40} {elapsed_tiered:>7.1f}s {n/elapsed_tiered:>9.1f}")
print(f"{'Docling (default)':<40} {elapsed_default:>7.1f}s {n/elapsed_default:>9.1f}")
print(f"{'Docling (force_full_page_ocr)':<40} {elapsed_force:>7.1f}s {n/elapsed_force:>9.1f}")

Mixed sample: 20 PDFs (10 digital + 10 scanned)

Method                                       Time   PDFs/sec
------------------------------------------------------------
PyMuPDF + Tesseract (per-page)               5.2s       3.8
Docling (default)                           13.7s       1.5
Docling (force_full_page_ocr)               25.5s       0.8


### The tradeoff

**Default Docling** is faster than forced OCR because it skips OCR on files that already have a text layer -- similar in spirit to the tiered strategy, but automatic. The downside: it trusts garbled text layers.

**Forced OCR** gives the best re-OCR results but is the slowest -- it re-OCRs every page regardless.

The **modular strategy** is fastest because PyMuPDF's `get_text()` is near-instant on clean files (~300 PDFs/sec), while combined packages still run layout analysis even when they don't need to OCR.

**GPU changes the calculus.** The speed comparisons above were run on CPU. Docling's layout models are deep learning models that run orders of magnitude faster on a CUDA GPU. With GPU acceleration, the speed gap shrinks dramatically -- not as fast as `get_text()`, but close enough that the layout benefits may outweigh the remaining difference. If you're working with a large corpus, a GPU instance or Google Colab can make Docling practical for bulk processing.

### Why not use a combined package for everything?

Could we skip PyMuPDF entirely and use Docling as the single tool — text-only mode for clean files, OCR mode for garbled ones? Let's test Docling with OCR disabled on clean PDFs to see if it can match PyMuPDF's speed:

In [12]:
# Docling with OCR disabled -- text extraction only
pipeline_options_text = PdfPipelineOptions()
pipeline_options_text.do_ocr = False
pipeline_options_text.do_table_structure = False
pipeline_options_text.generate_picture_images = False

converter_text = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options_text
        )
    }
)

# 50 digital PDFs (no page images)
clean_sample = []
for p in pdf_files:
    doc = pymupdf.open(p)
    if not doc[0].get_images():
        clean_sample.append(p)
    doc.close()
    if len(clean_sample) >= 50:
        break

# Docling text-only
start = time.perf_counter()
for p in clean_sample:
    result = converter_text.convert(str(p))
    _ = result.document.export_to_text()
elapsed_docling = time.perf_counter() - start

# PyMuPDF get_text()
start = time.perf_counter()
for p in clean_sample:
    doc = pymupdf.open(p)
    _ = "\n\n".join(page.get_text() for page in doc)
    doc.close()
elapsed_pymupdf = time.perf_counter() - start

n = len(clean_sample)
rate_pymupdf = n / elapsed_pymupdf
rate_docling = n / elapsed_docling
ratio = int(rate_pymupdf / rate_docling)

print(f"Text-only extraction on {n} digital PDFs:")
print()
print(f"{'Method':<35} {'Time':>8} {'PDFs/sec':>10}")
print("-" * 55)
print(f"{'PyMuPDF get_text()':<35} {elapsed_pymupdf:>7.2f}s {rate_pymupdf:>9.1f}")
print(f"{'Docling (do_ocr=False)':<35} {elapsed_docling:>7.2f}s {rate_docling:>9.1f}")
print(f"\nPyMuPDF is ~{ratio}x faster -- even with Docling's OCR disabled.")

Text-only extraction on 50 digital PDFs:

Method                                  Time   PDFs/sec
-------------------------------------------------------
PyMuPDF get_text()                     0.15s     327.8
Docling (do_ocr=False)                12.02s       4.2

PyMuPDF is ~78x faster -- even with Docling's OCR disabled.


The bottleneck isn't OCR -- it's **layout analysis**. Combined packages like Docling run their document understanding pipeline (layout detection, reading order, structure) on every page regardless of settings. That's what adds layout awareness, but it also means they can't compete with a lightweight reader like PyMuPDF for raw text extraction.

A "Docling-only tiered strategy" (text-only for clean files, OCR for garbled) would still be much slower than PyMuPDF + Tesseract, because the fast tier would take roughly 15--20 minutes on the clean files instead of ~7 seconds. This is a fundamental characteristic of the category, not a limitation specific to Docling.

## 9. Modular vs combined: when to use each

**Choose a combined package (Docling, Unstructured, etc.) when:**
- You want a single tool for the entire extraction process -- no tiered strategy
- Your documents have complex layouts (multi-column, tables, mixed content)
- Your dataset is small enough that speed doesn't matter
- You want layout-aware extraction and are willing to pay the speed cost

**Choose modular tools (PyMuPDF + Tesseract) when:**
- You're processing thousands of documents and speed matters
- Your documents are simple (single-column emails like ours)
- You need fine-grained control over each step
- You need cross-platform consistency

For this workshop, we'll use PyMuPDF + Tesseract for the full corpus extraction -- speed matters at 5,000 files. But combined packages like Docling are a strong choice for smaller datasets or complex layouts where you need layout awareness without building a tiered strategy yourself.

## Summary

- **Combined extraction packages** handle text extraction, OCR, and layout analysis in a single pipeline -- no tiered strategy needed
- Docling, Unstructured, Marker, and cloud APIs (Azure Document Intelligence, Google Document AI) are popular options
- Using Docling as an example: it analyzes page layout, reading order, and structure -- not just the text layer
- In **default mode**, it automatically reads text layers or OCRs as needed -- one tool, one code path
- Use `force_full_page_ocr=True` when you don't trust existing text layers (garbled OCR)
- **Slower** than the modular PyMuPDF + Tesseract strategy -- layout analysis runs on every page, even with OCR disabled. GPU acceleration closes this gap significantly.
- Each OCR engine has its own error profile -- combined packages add layout awareness, not guaranteed higher quality on every file
- Best suited for **smaller datasets**, **complex layouts**, or when you want **one simple code path**

**Next:** We'll look at the structure of these email PDFs and how to parse the extracted text into structured fields.